## 1주차: Basic

### 과제 1
`temp_samples = [72, 75, 79, 88, 91, 85]` 리스트를 만들고, **85 초과 값만** 출력하세요.

### 과제 2
`is_overheat(temp)` 함수를 작성하고, `temp >= 90`이면 `True`, 아니면 `False`를 반환하게 만드세요.

### 과제 3
NumPy를 사용해 `temp_samples`의 최솟값/최댓값/평균을 계산하세요.

In [14]:
# 1
temp_samples = [72, 75, 79, 88, 91, 85]
# print([tmp for tmp in temp_samples if tmp > 85])
for temp in temp_samples:
    if temp > 85: print(temp)
# 2
def is_overheat(temp): return temp >= 90

# 3
import numpy as np
nptemp_samples = np.array(temp_samples)
print(nptemp_samples.min(), nptemp_samples.max(), nptemp_samples.mean())


88
91
72 91 81.66666666666667


### 미니 미션

아래 조건을 만족하는 코드를 작성해보세요.

- `speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])`
- 속도 50 이상만 추출
- 추출된 값의 평균 계산
- 평균이 60 이상이면 `"고속 주행 구간"` 출력, 아니면 `"일반 주행 구간"` 출력

추가 도전:
- 인덱스까지 함께 출력해 어떤 시점에서 50 이상이었는지 확인

In [15]:
import numpy as np

speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])
if speed[speed >= 50].mean() >= 60: print("고속 주행 구간")
else : print("일반 주행 구간")
print("위치:", np.where(speed >= 50)[0]) # np.where는 튜플로 반환하므로 [0]으로 위치 배열을 추출

# idx, = np.where(speed >= 50)
# print("위치:", idx)

고속 주행 구간
위치: [2 4 5 6 7]


## 2주차: IMU

### 필터링 기법
1. **Average Filter**: 전체 데이터의 평균값으로 필터링
2. **Moving Average Filter**: 슬라이딩 윈도우를 사용한 이동 평균
3. **Exponential Moving Average Filter**: 가중 평균을 사용한 지수 이동 평균

In [16]:

class MotionSensorProcessor:
    """차량 모션 센서 데이터 처리 클래스"""

    def __init__(self):
        self.data = {}

    # 1. 평균 필터
    def average_filter(self, data: np.ndarray) -> np.ndarray: # 변수명은 ndarray임. np.array([])는 전환해주는 함수

        filtered_data = np.zeros_like(data)
        for i in range(len(data)):
            filtered_data[i] = data[:i+1].mean() #numpy 확실하니까 가능.
            # filtered_data[i] = np.mean(data[:i+1])

        return filtered_data
    
    # 2. 이동 평균 필터    
    def mov_avg_filter(self, data:np.ndarray, win_size: int =5) -> np.ndarray:
        if win_size <= 0: raise ValueError("윈도우 크기는 양수여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float) 
        # .mean()이 float을 뱉어도 filtered_data가 float여야 함. 
        # 만약 data가 int면 int로 저장됨
        for i in range(len(data)):
            start_idx = max(0, i + 1 - win_size)
            end_idx = i+1
            filtered_data[i] = data[start_idx:end_idx].mean()
        return filtered_data
    
    # 3. 지수 이동 평균 필터
    def exp_mov_avg_filter(self, data: np.ndarray, alpha: float = 0.3) -> np.ndarray:

        if alpha <= 0 or alpha > 1: raise ValueError("alpha 값은 0과 1 사이여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float)
        filtered_data[0] = data[0]
        
        for i in range(1, len(data)):
            filtered_data[i] = (1 - alpha) * filtered_data[i-1] + alpha * data[i]
        return filtered_data

### 3. Pulse Counting Method

일정 시간 동안의 펄스 개수로 속도 추정

**M-Method**: N = 펄스 개수, ΔT = 고정 시간 간격

### 수식
```
ω = 2π × N / (Z × ΔT)
```
- ω: 각속도 (rad/s)
- N: 샘플링 주기 동안의 펄스 수
- Z: 회전당 펄스 수 (pulses per revolution)
- ΔT: 샘플링 주기 (sec)


In [ ]:
def pulse_counting_method(self, time:np.ndarray, pulses:np.ndarray, pulses_per_revolution: int, counting_time_interval: float) ->np.ndarray:

	est_ang_vel = np.zeros(len(time))

	tmp_pulse_count = 0
	curr_est_ang_vel = 0.0
	prev_time = time[0] # 미리 초기화
	prev_pulse =pulses[0] # 미리 초기화

	for idx in range(1, len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: # 스칼라간의 비교는 'and'
			tmp_pulse_count += 1

		delta_time = time[idx] - prev_time
		
		if delta_time >= counting_time_interval:
			curr_est_ang_vel = tmp_pulse_count * 2 * np.pi / (pulses_per_revolution * counting_time_interval)
			tmp_pulse_count = 0
			prev_time = time[idx]
		 
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]
		
	return est_ang_vel

### 4. Pulse Timing Method

펄스 간 시간 간격으로 속도 추정

**T-Method**: N = 1, ΔT = 펄스 간 시간 간격  

### 수식
```
ω = 2π / (Z × ΔTd)
```
- ω: 각속도 (rad/s)
- Z: 회전당 펄스 수 (pulses per revolution)
- ΔTd: 펄스 간격


In [ ]:
def pulse_timing_method(self, time: np.ndarray, pulses: np.ndarray, pulses_per_revolution: int) ->np.ndarray:

	prev_time = time[0]
	prev_pulse = pulses[0]
	curr_est_ang_vel = 0.0
	est_ang_vel = np.zeros(len(time))

	for idx in range(1,len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: 
			delta_time = time[idx] - prev_time

			if delta_time < 1e-3:
				delta_time = 1e-3

			curr_est_ang_vel = 2 * np.pi / (pulses_per_revolution * delta_time)
			prev_time = time[idx]
	
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]

	return est_ang_vel

### 5. DR(Dead Reckoning)을 통한 위치 추정

Dead Reckoning(DR): **IMU 센서(자이로의 yaw rate)와 속도 정보(Wheel Odometry를 통해 취득)를 활용하여 차량의 위치를 적분적으로 추정하는 방법**
GPS/INS 데이터로부터 얻은 초기 위치와 속도를 바탕으로, IMU yaw rate와 속도를 적분하여 DR 경로를 계산

**장점**: GPS 없이도 연속적으로 위치 추정 가능

**단점**: 센서 노이즈와 바이어스로 인해 시간이 지남에 따라 누적 오차 발생

### DR 위치 추정 과정
1. **센서 데이터 동기화**  
   - IMU 시간축 기준으로 Wheel Odometry 속도 데이터를 보간(interpolation)  
   - `np.interp()`를 사용하여 imu_time 기준 속도 배열 생성  

2. **초기 상태 설정**  
   - 시작 위도, 경도, 방위각(azimuth) 추출  
   - 지구 반경 (R = 6378137m) 사용하여 위도/경도 변화량 계산 준비  

3. **방향(Heading) 업데이트**  
   - IMU yaw rate $(\omega)$ 적분  
   
   $\theta_k = \theta_{k-1} + \omega \cdot \Delta t$  

4. **변위 계산**  
   - 선속도 ($v$) 와 방향 $(\theta)$ 를 이용한 이동 거리 계산  
   
   $dx = v \cdot \cos(\theta) \cdot \Delta t$  
   
   $dy = v \cdot \sin(\theta) \cdot \Delta t$  

5. **위도/경도 업데이트**  
   - 지구 반경을 고려한 위도, 경도 증분 계산  
   
   $\Delta \phi = \frac{dy}{R}, \quad$
   $\Delta \lambda = \frac{dx}{R \cdot \cos(\phi)}$
     
   - 최종 위치 누적

   $\phi_k = \phi_{k-1} + \Delta \phi, \quad$
   $\lambda_k = \lambda_{k-1} + \Delta \lambda$


In [ ]:
    def calculate_dr_trajectory(self, imu_data: dict, inspvax_data: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        # 데이터 추출 및 시간 정규화
        imu_time = np.array(imu_data['time'])
        imu_yaw_rate = np.array([d.angular_velocity.z for d in imu_data['data']])
        inspvax_time = np.array(inspvax_data['time'])
        north_vel = np.array([d.north_velocity for d in inspvax_data['data']])
        east_vel = np.array([d.east_velocity for d in inspvax_data['data']])
        gt_velocity_raw = np.sqrt(north_vel**2 + east_vel**2)
        
        # 시간 동기화
        start_time = imu_time[0]
        imu_time -= start_time
        inspvax_time -= start_time

        velocity_interp = np.interp(imu_time, inspvax_time, gt_velocity_raw)

        # DR 위치 추정 및 초기 위치 설정 (위도/경도 직접 계산)
        R = 6378137.0  # 지구 반지름 (m)

        # GPS에서 도로 받음 → 라디안으로 변환 → 삼각함수로 계산 → 다시 도로 저장
        lat_rad = np.deg2rad(inspvax_data['data'][0].latitude)
        lon_rad = np.deg2rad(inspvax_data['data'][0].longitude)
        theta = np.deg2rad(90.0 - inspvax_data['data'][0].azimuth)

        # DR 위치 저장 ndarrays
        N = len(imu_time)
        dr_lat=np.zeros(N)
        dr_lon=np.zeros(N)
        dr_lat[0] = np.rad2deg(lat_rad)
        dr_lon[0] = np.rad2deg(lon_rad)

        for i in range(1, len(imu_time)):
            
            dt = imu_time[i] - imu_time[i-1]

            theta += imu_yaw_rate[i] * dt # 방위각 업데이트     
            dx = velocity_interp[i] * np.cos(theta) * dt # x축 이동 거리 업데이트
            dy = velocity_interp[i] * np.sin(theta) * dt # y축 이동 거리 업데이트
                       
            # 위도, 경도 업데이트
            lat_rad += dy / R
            lon_rad += dx / (R * np.cos(lat_rad))
            
            # DR 위치 저장
            dr_lat[i] = np.rad2deg(lat_rad)
            dr_lon[i] = np.rad2deg(lon_rad)
        
        return dr_lat, dr_lon, imu_time

## 3주차:GNSS

### 1. NMEA 데이터 분석

NMEA (National Marine Electronics Association) 0183은 GPS 수신기에서 가장 널리 사용되는 표준 통신 프로토콜입니다. 
이번 실습에서는 GGA와 RMC 메시지를 중심으로 NMEA 데이터를 분석합니다.

### NMEA 메시지 종류
- **GGA (Global Positioning System Fix Data)**: 위치, 고도, HDOP, 위성 수 등
- **RMC (Recommended Minimum)**: 위치, 속도, 방향, 날짜, 상태 등

In [ ]:
    def _parse_gga(self, line: str) -> Optional[Dict]:
        """
        GGA 메세지 구조: $GPGGA,<time>,<latitude>,<latitude_hemisphere>,<longitude>,<longitude_hemisphere>,<fix_quality>,<num_satellites>,<hdop>,<altitude>,<altitude_unit>,<geoid_height>,<checksum>
        """
        try:
            parts = line.split(',')
            if len(parts) < 15:
                return None
                
            # 위도 변환
            lat_str = parts[2] # 위도 문자열 추출 - parts 사용
            lat_hemisphere = parts[3] # 위도 방향 추출
            if lat_str and lat_hemisphere:
                lat_deg = int(lat_str[:2])
                lat_min = float(lat_str[2:])
                latitude = lat_deg + lat_min / 60.0
                if lat_hemisphere == 'S':
                    latitude = -latitude
            else:
                latitude = None
                
            # 경도 변환
            lon_str = parts[4] # 경도 문자열 추출
            lon_hemisphere = parts[5] # 경도 방향 추출
            if lon_str and lon_hemisphere:
                lon_deg = int(lon_str[:3])
                lon_min = float(lon_str[3:])
                longitude = lon_deg + lon_min / 60.0
                if lon_hemisphere == 'W':
                    longitude = -longitude
            else:
                longitude = None
                
            return {
                'time': parts[1], # UTC 시간 추출
                'latitude': latitude,
                'longitude': longitude,
                'fix_quality': int(parts[6]) if parts[6] else 0, # FIX 품질 추출
                'num_satellites': int(parts[7]) if parts[7] else 0, # 위성 수 추출
                'hdop': float(parts[8]) if parts[8] else 0.0, # HDOP 추출
                'altitude': float(parts[9]) if parts[9] else 0.0, # 고도 추출
                'altitude_unit': parts[10] if parts[10] else '', # 고도 단위 추출
                'geoid_height': float(parts[11]) if parts[11] else 0.0 # 지구 고도 추출 (undulation)
            }
        except (ValueError, IndexError):
            return None
    
    def _parse_rmc(self, line: str) -> Optional[Dict]:
        """
        RMC 메세지 구조: $GPRMC,<time>,<status>,<latitude>,<latitude_hemisphere>,<longitude>,<longitude_hemisphere>,<speed>,<course>
        ,<date>,<magnetic_variation>,<magnetic_variation_hemisphere>,<mode>,<checksum>
        """
        try:
            parts = line.split(',')
            if len(parts) < 12:
                return None
                
            # 위도 변환
            lat_str = parts[3] # 위도 문자열 추출
            lat_hemisphere = parts[4] # 위도 방향 추출
            if lat_str and lat_hemisphere:
                lat_deg = int(lat_str[:2])
                lat_min = float(lat_str[2:])
                latitude = lat_deg + lat_min / 60.0
                if lat_hemisphere == 'S':
                    latitude = -latitude
            else:
                latitude = None
                
            # 경도 변환
            lon_str = parts[5] # 경도 문자열 추출
            lon_hemisphere = parts[6] # 경도 방향 추출
            if lon_str and lon_hemisphere:
                lon_deg = int(lon_str[:3])
                lon_min = float(lon_str[3:])
                longitude = lon_deg + lon_min / 60.0
                if lon_hemisphere == 'W':
                    longitude = -longitude
            else:
                longitude = None
                
            return {
                'time': parts[1], # UTC 시간 추출
                'status': parts[2], # 상태 추출
                'latitude': latitude,
                'longitude': longitude,
                'speed_knots': float(parts[7]) if parts[7] else 0.0, # 속도 추출
                'course': float(parts[8]) if parts[8] else 0.0, # 방향 추출 (Track true)
                'date': parts[9] # 날짜 추출
            }
        except (ValueError, IndexError):
            return None

### 4. 좌표 변환 실습 (WGS84 ↔ ENU)

전역 좌표계(WGS84)와 지역 좌표계(ENU) 간 변환

### 좌표계 설명
- **WGS84**: 세계 측지계, GPS가 사용하는 전역 좌표계 (위도, 경도, 고도)
- **ENU**: East-North-Up, 지역 직교 좌표계 (동쪽, 북쪽, 위쪽)

### 변환 과정
1. **WGS84 → ENU**: 글로벌 좌표를 로컬 좌표로 변환
2. **ENU → WGS84**: 로컬 좌표를 다시 글로벌 좌표로 역변환
3. **정확도 검증**: 원본과 재변환 결과 비교

### 변환 수식
**WGS84 → ENU:**
```
E = Δlon × (N + h) × cos(φ₀)
N = Δlat × (M + h)  
U = Δh
```

**ENU → WGS84:**
```
Δlat = N / (M + h)
Δlon = E / ((N + h) × cos(φ₀))
Δh = U
```

여기서 M은 자오선 곡률반지름, N은 수직 곡률반지름입니다.

In [ ]:
    """
    WGS84 to ENU 변환에서 위도 경도 차이 계산시 실제 고도인 height를 사용하는데,
    ENU to WGS84 변환에서 위도 경도 차이 계산시 실제 고도는 직접 계산해야하기 때문에 ref_height를 사용해서 근사하고 있음.
    근데 이것은 수학적으로 완전한 역변환 관계가 아님. 이에 둘다 기준 고도인 ref_height를 사용하도록 수정이 필요한가? 아니면
    ENU to WGS84에서도 enu[:, 2]에 delta_height를 더해서 구한 실제 고도 height를 사용해서 둘다 height 사용?
    또는 한 쪽만?
    """
    
    # 1. WGS84 좌표 to ENU 좌표
    
    def wgs84_to_enu(self, llh: np.ndarray, ref_llh: np.ndarray) -> np.ndarray:

        enu = np.zeros_like(llh, dtype=float)
        
        ref_height = ref_llh[2]
        ref_lat = np.deg2rad(ref_llh[0])
        ref_lon = np.deg2rad(ref_llh[1])
        
        meridional_r = self.meridional_radius(ref_llh[0])
        normal_r = self.normal_radius(ref_llh[0])
        
        lat = np.deg2rad(llh[:, 0])
        lon = np.deg2rad(llh[:, 1])
        height = llh[:, 2]

        delta_lat_rad = lat - ref_lat # 위도 차이 계산
        delta_lon_rad = lon - ref_lon # 경도 차이 계산
        delta_height = height - ref_height # 고도 차이 계산

        enu[:, 0] = delta_lon_rad * (normal_r + ref_height) * np.cos(ref_lat) # East 추출
        enu[:, 1] = delta_lat_rad * (meridional_r + ref_height) # North 추출
        enu[:, 2] = delta_height # Up 추출

        return enu

    # 2. ENU 좌표 to WGS84 좌표
    
    def enu_to_wgs84(self, enu: np.ndarray, ref_llh: np.ndarray) -> np.ndarray:

        llh = np.zeros_like(enu, dtype=float)
        
        ref_height = ref_llh[2]
        ref_lat = np.deg2rad(ref_llh[0])
        ref_lon = np.deg2rad(ref_llh[1])
        
        meridional_r = self.meridional_radius(ref_llh[0]) # meridional_radius 함수 완성
        normal_r = self.normal_radius(ref_llh[0]) # normal_radius 함수 완성
        
        delta_lat_rad = enu[:, 1] / (meridional_r + ref_height) # 위도 차이 계산
        delta_lon_rad = enu[:, 0] / ((normal_r + ref_height) * np.cos(ref_lat)) # 경도 차이 계산
        height = enu[:, 2] + ref_height # 고도 계산

        llh[:, 0] = np.rad2deg(ref_lat + delta_lat_rad) # Latitude 추출
        llh[:, 1] = np.rad2deg(ref_lon + delta_lon_rad) # Longitude 추출
        llh[:, 2] = height # Height 추출

        return llh

## 4주차: LiDAR 

LiDAR 파트는 `custom_lidar.py`의 TODO 함수들을 흐름대로 정리하면 크게 다음 4단계로 볼 수 있습니다.

- **ROI Extraction**: 관심 있는 공간 범위만 남기기
- **k-NN Search**: 쿼리 점과 가까운 이웃 찾기
- **Least Squares Fitting**: 직선과 구를 최소 자승법으로 추정하기
- **RANSAC Helpers**: 샘플링, 모델 추정, 거리 계산, 인라이어 분류


In [ ]:
import numpy as np
import open3d as o3d
from typing import Tuple


### 1. ROI(Region of Interest) 추출

포인트 클라우드 전체를 다 쓰지 않고, 필요한 범위만 남기는 단계입니다.

핵심 아이디어는 축별 조건 마스크를 만든 뒤, 이를 AND 연산으로 합치는 것입니다.

- `x_mask`: X축 범위 필터
- `y_mask`: Y축 범위 필터
- `z_mask`: Z축 범위 필터
- `roi_mask = x_mask & y_mask & z_mask`


In [ ]:
def custom_extract_roi(
    pcd: o3d.geometry.PointCloud,
    x_range: Tuple[float, float] = (-20, 20),
    y_range: Tuple[float, float] = (-20, 20),
    z_range: Tuple[float, float] = (-3, 3),
) -> Tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud]:
    points = np.array(pcd.points)

    x_mask = (points[:, 0] >= x_range[0]) & (points[:, 0] <= x_range[1])
    y_mask = (points[:, 1] >= y_range[0]) & (points[:, 1] <= y_range[1])
    z_mask = (points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1])

    roi_mask = x_mask & y_mask & z_mask
    roi_points = points[roi_mask]
    roi_outliers = points[~roi_mask]

    roi_pcd = o3d.geometry.PointCloud()
    roi_pcd.points = o3d.utility.Vector3dVector(roi_points)

    if len(pcd.colors) > 0:
        colors = np.array(pcd.colors)
        roi_pcd.colors = o3d.utility.Vector3dVector(colors[roi_mask])

    roi_outliers_pcd = o3d.geometry.PointCloud()
    roi_outliers_pcd.points = o3d.utility.Vector3dVector(roi_outliers)

    if len(pcd.colors) > 0:
        colors = np.array(pcd.colors)
        roi_outliers_pcd.colors = o3d.utility.Vector3dVector(colors[~roi_mask])

    return roi_pcd, roi_outliers_pcd


### 2. Brute-force k-NN 검색

KD-Tree를 쓰기 전 가장 단순한 최근접 이웃 탐색입니다.

과정은 다음과 같습니다.

1. 모든 점과 쿼리 점 사이 거리 계산
2. 거리 기준 정렬
3. 앞에서부터 `k`개 선택


In [ ]:
def custom_brute_force_search(
    pcd: o3d.geometry.PointCloud,
    query: np.ndarray,
    k: int,
) -> Tuple[np.ndarray, np.ndarray]:
    points = np.array(pcd.points)
    query = np.array(query)

    distances = np.linalg.norm(points - query, axis=1)
    sorted_indices = np.argsort(distances)

    k_indices = sorted_indices[:k]
    k_distances = distances[k_indices]
    return k_indices, k_distances


### 3. Least Squares 직선 피팅

직선 방정식 `y = ax + b`를 최소 자승법으로 추정합니다.

행렬 형태로 쓰면 `H @ params = b` 이고, 정규 방정식은 아래와 같습니다.

```
params = (H^T H)^(-1) H^T b
```

여기서 `params = [a, b]^T` 입니다.


In [ ]:
def custom_least_squares_line_fitting(pcd: o3d.geometry.PointCloud) -> np.ndarray:
    points = np.array(pcd.points)
    x, y = points[:, 0], points[:, 1]
    n = len(points)

    H = np.column_stack([x, np.ones(n)])
    b = y

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    return inv_HtH @ Ht_b


### 4. Least Squares 구 피팅

구 방정식은 비선형이지만, 전개 후 선형 형태로 바꿔 최소 자승법을 적용할 수 있습니다.

- 중심: `(cx, cy, cz)`
- 반지름: `r`
- 마지막에 `radius_squared = cx^2 + cy^2 + cz^2 + d` 로 반지름 복원


In [ ]:
def custom_least_squares_sphere_fitting(
    pcd: o3d.geometry.PointCloud,
) -> Tuple[np.ndarray, float]:
    points = np.array(pcd.points)
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    n = len(points)

    H = np.column_stack([x, y, z, np.ones(n)])
    b = x ** 2 + y ** 2 + z ** 2

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    params = inv_HtH @ Ht_b
    cx, cy, cz = params[0] / 2, params[1] / 2, params[2] / 2
    d = params[3]

    radius_squared = cx ** 2 + cy ** 2 + cz ** 2 + d
    radius = np.sqrt(max(0, radius_squared))
    center = np.array([cx, cy, cz], dtype=float)
    return center, radius


### 5. RANSAC용 샘플링과 모델 추정

RANSAC은 매 반복마다 일부 점만 뽑아서 임시 모델을 만들고, 그 모델이 얼마나 많은 인라이어를 가지는지 평가합니다.

여기서는 다음 보조 함수들이 필요합니다.

- `custom_ransac_sample_points`: 중복 없이 랜덤 샘플링
- `custom_fit_line_from_samples`: 샘플 점들로 직선 추정
- `custom_fit_sphere_from_samples`: 샘플 점들로 구 추정


In [ ]:
def custom_ransac_sample_points(
    pcd: o3d.geometry.PointCloud,
    n_samples: int,
) -> Tuple[np.ndarray, np.ndarray]:
    points = np.array(pcd.points)
    n_points = len(points)

    sampled_indices = np.random.choice(range(n_points), size=n_samples, replace=False)
    sampled_points = points[sampled_indices]
    return sampled_points, sampled_indices


def custom_fit_line_from_samples(sampled_points: np.ndarray) -> np.ndarray:
    if len(sampled_points) < 2:
        raise ValueError("직선 피팅을 위해서는 최소 2개의 포인트가 필요합니다.")

    x, y = sampled_points[:, 0], sampled_points[:, 1]
    n = len(sampled_points)

    H = np.column_stack([x, np.ones(n)])
    b = y

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    return inv_HtH @ Ht_b


def custom_fit_sphere_from_samples(
    sampled_points: np.ndarray,
) -> Tuple[np.ndarray, float]:
    if len(sampled_points) < 4:
        raise ValueError("구 피팅을 위해서는 최소 4개의 포인트가 필요합니다.")

    x, y, z = sampled_points[:, 0], sampled_points[:, 1], sampled_points[:, 2]
    n = len(sampled_points)

    H = np.column_stack([x, y, z, np.ones(n)])
    b = x ** 2 + y ** 2 + z ** 2

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    params = inv_HtH @ Ht_b
    cx, cy, cz = params[0] / 2, params[1] / 2, params[2] / 2
    d = params[3]

    radius_squared = cx ** 2 + cy ** 2 + cz ** 2 + d
    radius = np.sqrt(max(0, radius_squared))
    center = np.array([cx, cy, cz], dtype=float)
    return center, radius


### 6. 거리 계산과 Inlier / Outlier 분류

RANSAC에서 모델을 하나 세웠다면, 이제 모든 점이 그 모델에 얼마나 잘 맞는지 거리로 평가합니다.

- 직선 거리: `|ax - y + b| / sqrt(a^2 + 1)`
- 구 거리: `|distance_to_center - radius|`
- threshold보다 작으면 inlier, 크거나 같으면 outlier


In [ ]:
def custom_compute_line_distances(
    pcd: o3d.geometry.PointCloud,
    line_params: np.ndarray,
) -> np.ndarray:
    points = np.array(pcd.points)
    x, y = points[:, 0], points[:, 1]
    a, b = line_params

    numerator = np.abs(a * x - y + b)
    denominator = np.sqrt(a ** 2 + 1)
    return numerator / denominator


def custom_compute_sphere_distances(
    pcd: o3d.geometry.PointCloud,
    center: np.ndarray,
    radius: float,
) -> np.ndarray:
    points = np.array(pcd.points)
    distances_to_center = np.linalg.norm(points - center, axis=1)
    return np.abs(distances_to_center - radius)


def custom_find_inliers_outliers(
    distances: np.ndarray,
    threshold: float,
) -> Tuple[np.ndarray, np.ndarray]:
    inlier_indices = np.where(distances < threshold)[0]
    outlier_indices = np.where(distances >= threshold)[0]
    return inlier_indices, outlier_indices
